## Ejemplo 2


In [1]:
# =================================================================
# A. PARÁMETROS DE TIEMPO Y FÍSICOS (No horarias)
# =================================================================

PARAMETROS_GENERALES = {
    # Capacidad MÁXIMA que el PPA permite usar en cualquier hora (MW)
    'PPA_MAX_CAPACIDAD': 10,
    # Potencia (capacidad) MÁXIMA requerida por el cliente (MW)
    'POTENCIA_REQUERIDA': 25,
    # Potencia que el contrato PPA ya cubre (MW)
    'POTENCIA_PPA_CUBIERTA': 8
}

# =================================================================
# B. PARÁMETROS DE COSTOS (Financieros)
# =================================================================

COSTOS = {
    # Costo fijo de la energía contratada por PPA ($/MWh)
    'COSTO_PPA': 650,
    # Costo de penalización por Déficit (no cubrir la demanda programada) ($/MWh)
    'COSTO_DEFICIT': 15000,
    # Costo de penalización por Excedente (programar de más) ($/MWh)
    'COSTO_EXCEDENTE': 500,
    # Costo de comprar Potencia Adicional en el Mercado de Potencia ($/MW-año)
    'COSTO_POTENCIA_ADICIONAL': 50000
}

In [2]:
# =================================================================
# C. DATOS HORARIOS (para t=1 a t=168)
# =================================================================

# Simulamos 168 horas de datos (ej. una semana completa)
HORAS = list(range(1, 169))

# 1. Demanda Horaria Esperada (MWh)
# Asumimos una demanda alta (25 MWh) en horas pico (8-22) y baja (15 MWh) en horas valle.
DEMANDA_HORARIA = {
    t: 25 if (t % 24 >= 8 and t % 24 <= 22) else 15
    for t in HORAS
}

# 2. Precio Marginal Local (PML) Proyectado ($/MWh)
# Asumimos precios SPOT muy altos en horas pico (t=18) para forzar el uso de PPA.
PRECIO_SPOT_HORARIO = {
    t: (1200 if t % 24 == 18 else 950)
    for t in HORAS
}

### Importamos librerías

In [3]:
import pyomo.environ as pe
import pyomo.opt as po

### Construimos el modelo

In [4]:
model= pe.ConcreteModel()

In [5]:
#Inicializamos los datos
model.horas= pe.Set(initialize=HORAS)

In [9]:
#Declaramos nuestras primeras variables
# Variables de Decisión
model.X_PPA = pe.Var(model.horas, domain=pe.NonNegativeReals)           # Energía PPA usada en t
model.X_SPOT = pe.Var(model.horas, domain=pe.NonNegativeReals)          # Energía SPOT comprada en t
model.Deficit = pe.Var(model.horas, domain=pe.NonNegativeReals)         # Desbalance no cubierto en t
model.Excedente = pe.Var(model.horas, domain=pe.NonNegativeReals)       # Energía programada de más en t
model.Potencia_Adquirida = pe.Var(domain=pe.NonNegativeReals)      # Potencia adicional comprada

'pyomo.core.base.var.IndexedVar'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.var.IndexedVar'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.var.IndexedVar'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
'pyomo.core.base.var.IndexedVar'>) on block unknown with a new Component
(type=<class 'pyomo.core.base.var.IndexedVar'>). This is usually indicative of
a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
(type=<class

In [10]:
#Definimos nuestras constricciones
def cons(model,t):
    return model.X_PPA[t]+model.X_SPOT[t]+model.Deficit[t]-model.Excedente[t] == DEMANDA_HORARIA[t]

model.constricion = pe.Constraint(model.horas, rule=cons)

def cons1(model,t):
    return model.X_PPA[t] <= PARAMETROS_GENERALES['PPA_MAX_CAPACIDAD']
model.constricion1 = pe.Constraint(model.horas, rule=cons1)

def cons2(model,t):
    return PARAMETROS_GENERALES['POTENCIA_PPA_CUBIERTA'] + model.Potencia_Adquirida >= PARAMETROS_GENERALES['POTENCIA_PPA_CUBIERTA']
model.constricion2 = pe.Constraint(model.horas, rule=cons2)

'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This
is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().
(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().



## Fórmulas del Modelo de Optimización de Cartera de Suministro

---

### 1. Variables de Decisión ($\mathbf{X}$)

Las siguientes variables son las que el modelo debe determinar. La mayoría son horarias ($\forall t \in \text{T}$), donde $\text{T}$ es el conjunto de las 168 horas de la semana.

* $\text{X\_PPA}_t$: Energía (MWh) comprada por PPA en la hora $t$.
* $\text{X\_SPOT}_t$: Energía (MWh) comprada en el Mercado Spot en la hora $t$.
* $\text{Déficit}_t$: Energía (MWh) no cubierta (incumplimiento) en la hora $t$.
* $\text{Excedente}_t$: Energía (MWh) programada de más en la hora $t$.
* $\text{Potencia\_Adquirida}$: Potencia adicional (MW) comprada para el año.

---

### 2. Función Objetivo: Minimización del Costo Total

El objetivo es minimizar la suma del costo de adquisición de energía, el costo de desbalance y el costo de potencia.

$$\min \quad \underbrace{\sum_{t \in \text{T}} [ (\text{X\_PPA}_t \cdot \text{COSTO\_PPA}) + (\text{X\_SPOT}_t \cdot \text{PRECIO\_SPOT}_t) ]}_{\text{Costo de Adquisición de Energía}}$$

$$+ \underbrace{\sum_{t \in \text{T}} [ (\text{Déficit}_t \cdot \text{COSTO\_DEFICIT}) + (\text{Excedente}_t \cdot \text{COSTO\_EXCEDENTE}) ]}_{\text{Costo de Desbalance (Penalizaciones)}}$$

$$+ \underbrace{(\text{Potencia\_Adquirida} \cdot \text{COSTO\_POTENCIA\_ADICIONAL})}_{\text{Costo de Potencia Anual}}$$

---

### 3. Restricciones ($\mathbf{S.A.}$)

#### R1. Restricción de Balance de Energía Horario
Garantiza que el suministro total (PPA + Spot) más/menos los desbalances iguale la demanda en cada hora.

$$\text{X\_PPA}_t + \text{X\_SPOT}_t + \text{Déficit}_t - \text{Excedente}_t = \text{DEMANDA\_HORARIA}_t \quad \forall t \in \text{T}$$

#### R2. Restricción de Capacidad Máxima del PPA
Asegura que el uso del PPA no exceda el límite contratado en ninguna hora.

$$\text{X\_PPA}_t \leq \text{PPA\_MAX\_CAPACIDAD} \quad \forall t \in \text{T}$$

#### R3. Restricción de Cumplimiento de Potencia
La potencia total disponible (la ya cubierta por PPA más la adicional comprada) debe ser igual o mayor que la potencia requerida por el cliente.

$$\text{POTENCIA\_PPA\_CUBIERTA} + \text{Potencia\_Adquirida} \geq \text{POTENCIA\_REQUERIDA}$$

In [11]:
#Función objetivo
def obj_rule(model):
    # 1. Costo de Adquisición de Energía (PPA + SPOT)
    costo_adquisicion = sum(
        (model.X_PPA[t] * COSTOS['COSTO_PPA']) +
        (model.X_SPOT[t] * PRECIO_SPOT_HORARIO[t])
        for t in model.horas
    )

    # 2. Costo de Desbalance (Déficit + Excedente)
    costo_desbalance = sum(
        (model.Deficit[t] * COSTOS['COSTO_DEFICIT']) +
        (model.Excedente[t] * COSTOS['COSTO_EXCEDENTE'])
        for t in model.horas
    )

    # 3. Costo de Potencia Adicional
    costo_potencia = model.Potencia_Adquirida * COSTOS['COSTO_POTENCIA_ADICIONAL']

    return costo_adquisicion + costo_desbalance + costo_potencia

model.CostoTotal = pe.Objective(rule=obj_rule, sense=pe.minimize)

### Resolver

In [12]:
solver = po.SolverFactory('glpk')
results = solver.solve(model, tee=True)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpobj25h70.glpk.raw
 --wglp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmp9op4a6vg.glpk.glp
 --cpxlp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpr_7p1nyo.pyomo.lp
Reading problem data from '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpr_7p1nyo.pyomo.lp'...
504 rows, 673 columns, 1008 non-zeros
3875 lines were read
Writing problem data to '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmp9op4a6vg.glpk.glp'...
3365 lines were written
GLPK Simplex Optimizer 5.0
504 rows, 673 columns, 1008 non-zeros
Preprocessing...
168 rows, 504 columns, 504 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 168
      0: obj =   2.320500000e+06 inf =   1.890e+03 (168)
    168: obj =   2.913750000e+06 inf =   0.000e+00 (0

In [17]:
import pandas as pd

# Extraer resultados en listas/diccionarios
resultados = {
    'Hora': [],
    'X_PPA (MWh)': [],
    'X_SPOT (MWh)': [],
    'Déficit (MWh)': [],
    'Excedente (MWh)': [],
    'Demanda (MWh)': [],
    'Precio Spot ($/MWh)': []
}

for t in model.horas:
    resultados['Hora'].append(t)
    resultados['X_PPA (MWh)'].append(pe.value(model.X_PPA[t]))
    resultados['X_SPOT (MWh)'].append(pe.value(model.X_SPOT[t]))
    resultados['Déficit (MWh)'].append(pe.value(model.Deficit[t]))
    resultados['Excedente (MWh)'].append(pe.value(model.Excedente[t]))
    resultados['Demanda (MWh)'].append(DEMANDA_HORARIA[t])
    resultados['Precio Spot ($/MWh)'].append(PRECIO_SPOT_HORARIO[t])

df_resultados = pd.DataFrame(resultados)

# Mostrar las primeras filas
print(" Resultados por hora:")
display(df_resultados.head(24))  # primeras 24 horas

# Mostrar Potencia adquirida y Costo total
print("\n Potencia adicional adquirida (MW):", pe.value(model.Potencia_Adquirida))
print(" Costo total ($):", pe.value(model.CostoTotal))


/Users/erickavendanogarcia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


 Resultados por hora:


,Hora,X_PPA (MWh),X_SPOT (MWh),Déficit (MWh),Excedente (MWh),Demanda (MWh),Precio Spot ($/MWh)
0,1,10.0,5.0,0.0,0.0,15,950
1,2,10.0,5.0,0.0,0.0,15,950
2,3,10.0,5.0,0.0,0.0,15,950
3,4,10.0,5.0,0.0,0.0,15,950
4,5,10.0,5.0,0.0,0.0,15,950
5,6,10.0,5.0,0.0,0.0,15,950
6,7,10.0,5.0,0.0,0.0,15,950
7,8,10.0,15.0,0.0,0.0,25,950
8,9,10.0,15.0,0.0,0.0,25,950
9,10,10.0,15.0,0.0,0.0,25,950



 Potencia adicional adquirida (MW): 0.0
 Costo total ($): 2913750.0
